# Phase Picking with SeisBench and EarthScope Data

This notebook is a short, self-contained introduction to **[SeisBench](https://github.com/seisbench/seisbench)** — a toolbox for machine-learning models in seismology — and walks through a single, concrete task: **detecting P and S arrivals on a real earthquake recording pulled from EarthScope's FDSN web service.**

**What you'll do**

1. Pull waveforms for a known earthquake from the EarthScope FDSN service with ObsPy.
2. Load a pretrained deep-learning phase picker (PhaseNet) through SeisBench's common model interface.
3. Run the picker and inspect the returned picks.
4. Overlay the predicted P/S arrival times on the waveforms.

**Why SeisBench?** Classical pickers (STA/LTA, AR-AIC) need careful per-station tuning. SeisBench wraps a family of trained neural-network pickers behind one API (`from_pretrained` → `classify`), so the same three lines of code run PhaseNet, EQTransformer, or any other supported model. Weights are downloaded and cached on first use.


## 1. Setup

These packages are in the geolab image (`obspy`, plus `seisbench` via pip). If you're running elsewhere, uncomment the install line.

In [ ]:
# %pip install seisbench obspy matplotlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from obspy import UTCDateTime
from obspy.clients.fdsn import Client

import seisbench
import seisbench.models as sbm

print("SeisBench version:", seisbench.__version__)

## 2. Pull waveforms from EarthScope

ObsPy ships a built-in shortcut, `Client("EARTHSCOPE")`, that points at EarthScope's FDSN web service (`https://service.iris.edu`) — no credentials needed for open data.

We'll use a well-recorded event so the picks are easy to interpret: the **2019-07-06 M7.1 Ridgecrest, California** mainshock, recorded at broadband station **CI.CCC** (a Southern California Seismic Network station relatively close to the rupture).

For phase picking you want the three broadband channels (`BH?` → BHZ, BHN/BH1, BHE/BH2). SeisBench handles the component grouping internally.

In [ ]:
client = Client("EARTHSCOPE")

# 2019-07-06 03:19:53 UTC -- M7.1 Ridgecrest mainshock origin time
origin_time = UTCDateTime("2019-07-06T03:19:53")

network  = "CI"
station  = "CCC"
location = "*"
channel  = "BH?"          # all three broadband components

# Grab a 120 s window starting ~10 s before origin
t0 = origin_time - 10
t1 = origin_time + 110

stream = client.get_waveforms(
    network, station, location, channel,
    starttime=t0, endtime=t1,
)
print(stream)

### Quick pre-processing

SeisBench pickers expect the raw three-component waveform; they apply their own normalization internally and **resample to the model's native sampling rate (100 Hz for PhaseNet) automatically**. So minimal prep is needed — we only:

- `merge` to heal any gaps,
- `detrend` to remove a linear trend.

We deliberately **do not** remove the instrument response or filter: the pretrained weights were trained on lightly-processed data, and over-processing can degrade picks. Let SeisBench do the normalization.

In [ ]:
stream.merge(method=1, fill_value="interpolate")
stream.detrend("linear")

# Plot the raw vertical component to see the event
st_z = stream.select(channel="BHZ")
fig = st_z.plot(handle=True, size=(900, 250))
plt.show()

## 3. Load a pretrained picker

The pattern is always the same:

```python
sbm.<Model>.list_pretrained()        # what weight sets exist
model = sbm.<Model>.from_pretrained("<weights>")
```

We'll use **PhaseNet** (Zhu & Beroza, 2019) with the **`instance`** weights (trained on the Italian INSTANCE dataset) — a solid general-purpose choice. Other options include `stead`, `scedc`, `ethz`, `geofon`, and `original`. Weights download once, then cache to `~/.seisbench/`.

In [ ]:
# See which pretrained weight sets are available for PhaseNet
print(sbm.PhaseNet.list_pretrained())

In [ ]:
# Load PhaseNet. update=True on first load applies a normalization fix
# noted in the SeisBench docs for several older weight versions.
model = sbm.PhaseNet.from_pretrained("instance", update=True)
model.eval()   # inference mode

print(model)
print("Native sampling rate:", model.sampling_rate, "Hz")
print("Phases predicted     :", model.labels)

## 4. Run the picker

`model.classify(stream)` does everything: windowing, resampling, normalization, the forward pass, peak-finding on the output probability curves, and thresholding. It returns a `ClassifyOutput` object whose `.picks` attribute is a list of `Pick` objects.

> **API note:** in current SeisBench you access picks via `output.picks` — you can no longer index `output[0]`. Each `Pick` carries `trace_id`, `phase`, `peak_time` (a `UTCDateTime`), and `peak_value` (the model confidence, 0–1).

The `P_threshold` / `S_threshold` arguments control sensitivity; the default is 0.3. Lower → more (and noisier) picks.

In [ ]:
output = model.classify(
    stream,
    P_threshold=0.3,
    S_threshold=0.3,
)

picks = output.picks
print(f"{len(picks)} pick(s) found\n")
for p in picks:
    print(f"  {p.phase}  {p.peak_time}  conf={p.peak_value:.3f}  ({p.trace_id})")

Convenient tabular view — `picks` can be turned into a DataFrame:

In [ ]:
df = picks.to_dataframe()
df

## 5. Visualize the picks

The most informative view overlays two things on a shared time axis:

1. the three-component waveforms, with vertical lines at each predicted arrival, and
2. the **probability curves** PhaseNet outputs — `model.annotate(stream)` returns these as an ObsPy `Stream` (one trace per phase), which is what `classify` peak-picks under the hood.

Seeing the probability traces makes it clear *why* a pick landed where it did.

In [ ]:
# Probability curves (the raw network output, before peak-picking)
preds = model.annotate(stream)
print(preds)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

# Reference time = window start, so x-axis is seconds since t0
ref = stream[0].stats.starttime

# --- top 3 panels: the three components ---
comp_order = ["Z", "N", "E", "1", "2"]
traces = sorted(stream, key=lambda tr: comp_order.index(tr.stats.channel[-1])
                if tr.stats.channel[-1] in comp_order else 99)

phase_colors = {"P": "tab:blue", "S": "tab:red"}

for ax, tr in zip(axes[:3], traces[:3]):
    times = tr.times(reftime=ref)
    ax.plot(times, tr.data, "k", lw=0.5)
    ax.set_ylabel(tr.stats.channel)
    for p in picks:
        c = phase_colors.get(p.phase, "tab:green")
        ax.axvline(p.peak_time - ref, color=c, lw=1.5, ls="--")

# --- bottom panel: probability curves ---
ax = axes[3]
for tr in preds:
    ph = tr.stats.channel.split("_")[-1]   # e.g. "PhaseNet_P" -> "P"
    if ph in ("P", "S"):
        ax.plot(tr.times(reftime=ref), tr.data,
                color=phase_colors.get(ph, "gray"), label=ph)
ax.set_ylabel("probability")
ax.set_xlabel(f"seconds since {ref.isoformat()}")
ax.set_ylim(0, 1)
ax.legend(loc="upper right")

# legend for the pick lines
for ph, c in phase_colors.items():
    axes[0].plot([], [], color=c, ls="--", label=f"{ph} pick")
axes[0].legend(loc="upper right")

fig.suptitle(f"PhaseNet picks — {network}.{station}  (Ridgecrest M7.1)")
fig.tight_layout()
plt.show()

## 6. Where to go next

- **Swap the model.** Replace one line to try EQTransformer, which also outputs event *detections* alongside picks:
  ```python
  model = sbm.EQTransformer.from_pretrained("instance", update=True)
  out = model.classify(stream)
  out.detections   # EQTransformer-only
  out.picks
  ```
- **Pick across a network.** `classify` accepts a multi-station stream and groups traces by station automatically — feed it a `get_waveforms_bulk` result to pick a whole array at once.
- **Build a catalog.** Pipe SeisBench picks into a phase associator (e.g. GaMMA or PyOcto) and then a locator to go from picks → events.
- **Compare weight sets.** Run `instance`, `stead`, and `scedc` on the same trace and compare confidences — a quick way to see how training-domain shift affects picks on your region's data.

### References
- Woollam et al. (2022), *SeisBench — A Toolbox for Machine Learning in Seismology*, SRL.
- Zhu & Beroza (2019), *PhaseNet: a deep-neural-network-based seismic arrival-time picking method*, GJI 216(1), 261–273.
